# SARBLOH E007: vLLM serving bench, Qwen3.8-Flash-Next and Gemma-4-31B (no games)

Every profile of each model runs on a fresh server with the same workloads (`harness/llm/bench.py`): tool-call check,
decode at 1 / 8 / 10 streams, cold 10-way prefill, an agent-shaped append-only conversation (prefix-cache hit rate),
the same with thinking on. Then the fastest profile of each model runs a stress test with the watchdog on while the
engine is frozen on purpose (SIGSTOP), which checks detection, stack dump, restart and client recovery.

The `harness/llm` source is embedded in cell 1 by `scripts/kaggle_push.py vllm-bench` (no agent dataset needed).
Inputs: the vLLM 0.19.0 wheelhouse, the Gemma overlay and weights, the Qwen3.8-27B FP8 snapshot, and for
Qwen3.8-Flash-Next NVFP4 the Kaggle model plus Keith Tyser's pinned vLLM runtime and serving bundle (v4).
Internet off. Outputs: `e007/vllm_bench.json`, `e007/server_events.jsonl`, `e007/vllm-server.log`.

## 0. CONFIG

In [ ]:
import time
NOTEBOOK_START = time.time()

BUDGET_S = 2.5 * 3600          # hard stop for the whole bench; each model gets a share below
STREAMS = 10                   # the agent runs 10 games at once
STRESS_MINUTES = 10.0          # per model, on its fastest profile; 0 = skip
PLAN = [                       # (model, profiles, share of the budget)
    # v4: Flash-Next only (v3 measured the 27B FP8 and Gemma profiles, stress tests included)
    ("qwen", ["qwen_flash", "qwen_flash_pc", "qwen_flash_pc_64k"], 1.0),
]
VLLM = {"wheelhouse_dataset": "banwait13/sarbloh-vllm-wheelhouse", "port": 8000, "startup_timeout_s": 1500.0,
        "max_inflight": 32, "watchdog_cfg": {}}
LLM_CFG = {"request_timeout_s": 900.0, "retries": 4}

## 1. Source (embedded at push time) and environment

In [ ]:
# @@PRIME_LLM_SOURCES@@
import subprocess, sys
from pathlib import Path

SRC = Path("/tmp/sarbloh-src")
for rel, text in SOURCES.items():
    p = SRC / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
sys.path.insert(0, str(SRC))
print("source:", SOURCES_META)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
# E007 v2 got 2x T4 (15 GB) and OOM'd on every profile: stop at once unless this is the ~96 GB RTX Pro 6000
_vram_mib = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True).stdout.split()[0])
assert _vram_mib > 90_000, f"wrong GPU: {_vram_mib} MiB; the bench needs the RTX Pro 6000 (machine_shape NvidiaRtxPro6000)"
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
print(subprocess.run("ls -la /kaggle/input /kaggle/input/* | head -60", shell=True, capture_output=True, text=True).stdout)

## 2. Bench

In [ ]:
import json, traceback
from harness.llm import bench

OUT = Path("/kaggle/working/e007")
ROWS = []
start = time.time()
for model, profiles, share in PLAN:
    budget_end = NOTEBOOK_START + BUDGET_S
    deadline = min(budget_end, time.time() + share * (budget_end - start))
    print(f"==== {model}: {profiles} until {time.ctime(deadline)}", flush=True)
    try:
        ROWS = bench.run(model, profiles, VLLM, LLM_CFG, OUT, deadline, streams=STREAMS,
                         stress_minutes=STRESS_MINUTES)
    except Exception:
        traceback.print_exc()

## 3. Summary

In [ ]:
rows = json.loads((OUT / "vllm_bench.json").read_text()) if (OUT / "vllm_bench.json").exists() else []
def g(r, k, f):
    v = r.get(k)
    return v.get(f) if isinstance(v, dict) else None
print(f"{'model':6} {'profile':18} {'start_s':>7} {'tool':>6} {'d1':>6} {'d8':>7} {'d10':>7} {'acc':>5} "
      f"{'prefill':>8} {'turn1':>6} {'turnN':>6} {'hit':>5}")
for r in rows:
    if "stress" in r:
        continue
    print(f"{r.get('model',''):6} {r.get('profile',''):18} {str(r.get('startup_s')):>7} {str(r.get('tool_mode')):>6} "
          f"{str(g(r,'decode_1','tok_s')):>6} {str(g(r,'decode_8','tok_s')):>7} {str(g(r,f'decode_{STREAMS}','tok_s')):>7} "
          f"{str(g(r,f'decode_{STREAMS}','spec_accept')):>5} {str(g(r,'cold_prefill','prefill_tok_s')):>8} "
          f"{str(g(r,'agent_like','turn1_mean_s')):>6} {str(g(r,'agent_like','later_turn_mean_s')):>6} "
          f"{str(g(r,'agent_like','prefix_hit_rate')):>5}  {r.get('error','')[:120]}")
for r in rows:
    if "stress" in r:
        print("STRESS", r["model"], r["profile"], json.dumps(r["stress"], default=str)[:1500])
ev = OUT / "server_events.jsonl"
if ev.exists():
    for line in ev.read_text().splitlines():
        e = json.loads(line)
        if e["event"] in ("freeze", "fallback", "give_up", "start_failed", "metrics_keys"):
            print(json.dumps(e)[:2500])
print("E007_RESULT " + json.dumps(rows, default=str)[:20000])
print(f"notebook wall: {time.time() - NOTEBOOK_START:.0f}s")